# 📚 Appendix B: Deep Modules

> *Optionales Notebook — für alle die verstehen wollen, warum verschiedene Module-Tiefen existieren.*

Dieses Notebook erklärt das **Deep Module**-Prinzip aus *A Philosophy of Software Design*: Gleiches Interface, verschiedene interne Komplexität.

# 🏛️ Deep Modules — Tiefe statt Oberfläche

## A Philosophy of Software Design für LLMs

John Ousterhout sagt: **Das beste Modul hat ein einfaches Interface und komplexe Innereien.** Dieses Prinzip der **Modultiefe** gilt genauso für KI-Systeme:

- **Flach** — ein LLM-Aufruf, direkte Antwort
- **Mittel** — ein LLM-Aufruf, aber mit automatischem Denkschritt (Reasoning)
- **Tief** — mehrere LLM-Aufrufe + Tool-Nutzung

Und das Geniale: **alle drei haben das gleiche Interface**: `module(**inputs) → prediction`

Probier's aus — du wirst den Unterschied sehen!

In [ ]:
from dspy_tasks.config import get_available_models, configure_dspy

# Verfügbare Modelle
print("Verfügbare Modelle:")
for m in get_available_models()[:10]:
    print(f"  • {m}")

# Modell wählen (ändere den String um ein anderes zu nutzen)
MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)
print(f"\n✅ Konfiguriert: {MODEL}")


In [ ]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "Predict", "detail": "1 Aufruf, direkt", "icon": "⬜", "color": "#8a8886"},
    {"label": "ChainOfThought", "detail": "1 Aufruf + Reasoning", "icon": "🟦", "color": "#0078d4"},
    {"label": "ReAct", "detail": "N Aufrufe + Tools", "icon": "🟦🟦", "color": "#005a9e"},
], title="Gleiche Schnittstelle, verschiedene Tiefe")

## Die drei Tiefen im Vergleich

Lass uns das gleiche Problem mit verschiedenen Modultiefen lösen. Probier's aus — für einfache Aufgaben reicht `Predict`. Aber sobald Nachdenken gefragt ist, macht `ChainOfThought` den Unterschied.

In [ ]:

# SHALLOW: Just predict
shallow = dspy.Predict(TranslateEnDe)
result_shallow = shallow(english_text="The cat sat on the mat while the dog chased its tail.")

# DEEP: Chain of Thought
deep = dspy.ChainOfThought(TranslateEnDe)
result_deep = deep(english_text="The cat sat on the mat while the dog chased its tail.")

print("Predict (flach):")
print(f"  → {result_shallow.german_text}")
print()
print("ChainOfThought (tief):")
if hasattr(result_deep, 'rationale'):
    print(f"  Reasoning: {str(result_deep.rationale)[:200]}")
print(f"  → {result_deep.german_text}")

## Wo Tiefe wirklich zählt

Für Sentiment-Analyse reicht ein flacher Aufruf locker. Aber bei Mathe-Aufgaben, wo man Schritt für Schritt rechnen muss, bringt ein Denkschritt den entscheidenden Unterschied — er wird **automatisch** hinzugefügt!

Schau dir die Scores an — **gleiche Aufgabe, gleiches Modell, aber verschiedene Modultiefe**:

In [ ]:
# Run math task with both module types
task = get_task("math_word")
_, devset = task.split_examples()
eval_set = devset[:8]

from dspy_tasks.calculations import numeric_match
from dspy_tasks.actions import _evaluate_examples, _mean

# Shallow
shallow_math = dspy.Predict(task.signature_class)
shallow_results = _evaluate_examples(shallow_math, eval_set, task.metric_fn)
shallow_score = _mean([r["score"] for r in shallow_results])

# Deep
deep_math = dspy.ChainOfThought(task.signature_class)
deep_results = _evaluate_examples(deep_math, eval_set, task.metric_fn)
deep_score = _mean([r["score"] for r in deep_results])

display_score("Predict (shallow)", shallow_score)
display_score("ChainOfThought (deep)", deep_score)
display_improvement(shallow_score, deep_score)

display_insight("Deep Module Lektion",
    f"Gleiches Interface, gleiche Aufgabe, gleiches Modell — aber ChainOfThought "
    f"erreicht {deep_score:.0%} vs Predict mit {shallow_score:.0%}. "
    "Die interne Tiefe des Moduls ermöglicht erst das Reasoning.")

In [ ]:
from dspy_tasks.config import configure_dspy
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in [get_task(tid) for tid in ["translation", "format_compliance", "math_word", "logical_deduction"]]],
    description="Task:")
module_dd = widgets.Dropdown(options=["Predict", "ChainOfThought"], description="Module:")
btn = run_button("Run")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        task = get_task(task_dd.value)
        if module_dd.value == "ChainOfThought":
            module = dspy.ChainOfThought(task.signature_class)
        else:
            module = dspy.Predict(task.signature_class)
        _, devset = task.split_examples()
        results = _evaluate_examples(module, devset[:8], task.metric_fn)
        score = _mean([r["score"] for r in results])
        display_score(f"{task.name} ({module_dd.value})", score)
        display_results_table(results[:5])

btn.on_click(on_run)
display(widgets.HBox([task_dd, module_dd, btn]), out)

## 📚 Weiterführend

Für den Hauptlernpfad (Evaluation → Optimierung → Domain-Tuning → Agenten) starte bei **Notebook 00**.